# Setup and Installation

In [ ]:
!pip install -q openai-whisper pydub tqdm
!apt-get install -q ffmpeg

print("✓ Dependencies installed")
print("Checking GPU availability...")
import torch
if torch.cuda.is_available():
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ No GPU available, will use CPU (slower)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
✓ Dependencies installed
Checking GPU availability...
✓ GPU available: Tesla T4


# Mount Google Drive (optional - for saving/loading files)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

Mounted at /content/drive
✓ Google Drive mounted at /content/drive


# Upload Audio Files

In [ ]:
from google.colab import files
import os

# Option A: Upload from your computer
print("Select audio files to upload...")
uploaded = files.upload()

# Move uploaded files to a working directory
os.makedirs('/content/audio_input', exist_ok=True)
for filename in uploaded.keys():
    os.rename(f'/content/{filename}', f'/content/audio_input/{filename}')
    print(f"✓ Uploaded: {filename}")

Select audio files to upload...


Saving 05. Kupecká knížata.mp3 to 05. Kupecká knížata.mp3
✓ Uploaded: 05. Kupecká knížata.mp3


# Transcription code

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict
import warnings
warnings.filterwarnings('ignore')

import whisper
from tqdm import tqdm


class WhisperTranscriber:

    def __init__(
        self,
        whisper_model: str = "large",
        language: str = "cs",
        output_dir: str = "output"
    ):
        self.whisper_model = whisper_model
        self.language = language
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        print(f"Loading Whisper model ({whisper_model})...")
        self.model = whisper.load_model(whisper_model)
        print("✓ Model loaded\n")

    def transcribe_audio(self, audio_path: str) -> Dict:
        """Transcribe audio file and cache the results."""
        audio_path = Path(audio_path)

        # Check for cached transcription
        cache_file = audio_path.parent / f"{audio_path.stem}.transcription.json"

        if cache_file.exists():
            try:
                print(f"Loading cached transcription from {cache_file.name}...")
                with open(cache_file, 'r', encoding='utf-8') as f:
                    cached_data = json.load(f)
                print(f"✓ Loaded from cache")
                return cached_data
            except Exception as e:
                print(f"⚠ Error loading cache: {e}, re-transcribing...")

        # Transcribe with Whisper
        print(f"Transcribing: {audio_path.name}")
        print("This may take a while depending on audio length...")

        result = self.model.transcribe(
            str(audio_path),
            language=self.language,
            word_timestamps=True,
            verbose=False
        )

        # Extract word-level data
        all_words = []
        for segment in result.get('segments', []):
            if 'words' in segment:
                all_words.extend(segment['words'])

        print(f"✓ Transcribed {len(all_words)} words")

        # Prepare transcription data
        transcription_data = {
            'text': result['text'],
            'words': all_words,
            'segments': result.get('segments', []),
            'language': self.language,
            'model': self.whisper_model,
            'audio_file': audio_path.name
        }

        # Save cache
        try:
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(transcription_data, f, ensure_ascii=False, indent=2)
            print(f"✓ Transcription cached to: {cache_file.name}")
        except Exception as e:
            print(f"⚠ Warning: Could not save cache: {e}")

        return transcription_data

    def save_transcription_output(self, transcription_data: Dict, audio_path: Path):
        """Save transcription in original JSON format."""
        base_name = audio_path.stem

        # Save JSON with word timestamps - SAME FORMAT AS ORIGINAL
        json_file = self.output_dir / f"{base_name}.transcription.json"
        try:
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(transcription_data, f, ensure_ascii=False, indent=2)
            print(f"✓ Saved JSON to: {json_file.name}")
        except Exception as e:
            print(f"✗ Error saving JSON: {e}")

    def process_file(self, audio_path: str) -> Dict:
        """Process a single audio file."""
        audio_path = Path(audio_path)

        print(f"\n{'='*60}")
        print(f"Processing: {audio_path.name}")
        print(f"{'='*60}\n")

        if not audio_path.exists():
            print(f"✗ Error: File not found: {audio_path}")
            return {'success': False, 'error': 'File not found'}

        try:
            # Transcribe
            transcription_data = self.transcribe_audio(str(audio_path))

            # Save outputs
            self.save_transcription_output(transcription_data, audio_path)

            # Stats
            word_count = len(transcription_data['words'])
            char_count = len(transcription_data['text'])

            print(f"\n{'='*60}")
            print(f"SUMMARY")
            print(f"{'='*60}")
            print(f"Words: {word_count}")
            print(f"Characters: {char_count}")
            print(f"Text: {transcription_data['text'][:100]}...")
            print(f"Output: {self.output_dir}")
            print(f"{'='*60}\n")

            return {
                'success': True,
                'word_count': word_count,
                'char_count': char_count,
                'text': transcription_data['text']
            }

        except Exception as e:
            print(f"✗ Error processing file: {e}")
            import traceback
            traceback.print_exc()
            return {'success': False, 'error': str(e)}

# Configure and Run Transcription

In [ ]:
# Configuration
CONFIG = {
    'whisper_model': 'large',      # Options: 'tiny', 'base', 'small', 'medium', 'large'
    'language': 'cs',               # Language code (e.g., 'en', 'cs', 'es', etc.)
    'output_dir': '/content/output'
}

# Initialize transcriber
transcriber = WhisperTranscriber(**CONFIG)

# Process all audio files
input_dir = '/content/audio_input'
audio_files = list(Path(input_dir).glob('*.wav')) + \
              list(Path(input_dir).glob('*.mp3')) + \
              list(Path(input_dir).glob('*.flac')) + \
              list(Path(input_dir).glob('*.m4a')) + \
              list(Path(input_dir).glob('*.ogg'))

print(f"Found {len(audio_files)} audio files to process\n")

if len(audio_files) == 0:
    print("⚠ No audio files found! Please upload files in Cell 3.")
else:
    results = []
    for audio_file in audio_files:
        result = transcriber.process_file(str(audio_file))
        results.append(result)

    print("\n" + "="*60)
    print("ALL FILES PROCESSED")
    print("="*60)
    successful = sum(1 for r in results if r.get('success'))
    print(f"Successfully transcribed: {successful}/{len(audio_files)} files")
    total_words = sum(r.get('word_count', 0) for r in results if r.get('success'))
    print(f"Total words transcribed: {total_words}")

Loading Whisper model (large)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 18.12 MiB is free. Process 3643 has 14.72 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 546.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Download Results

In [ ]:
# Option A: Download as ZIP
import shutil
print("Creating ZIP archive of transcriptions...")
shutil.make_archive('/content/transcriptions', 'zip', '/content/output')
files.download('/content/transcriptions.zip')
print("✓ Download started!")

# Option B: Copy to Google Drive (uncomment if preferred)
# import shutil
# output_drive_path = '/content/drive/MyDrive/transcriptions'
# shutil.copytree('/content/output', output_drive_path, dirs_exist_ok=True)
# print(f"✓ Results copied to Google Drive: {output_drive_path}")

Creating ZIP archive of transcriptions...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download started!


# View Results

In [ ]:
# Display transcription results in Colab
from pathlib import Path

output_dir = Path('/content/output')
json_files = list(output_dir.glob('*.transcription.json'))

print(f"Found {len(json_files)} transcription files:\n")

for json_file in json_files:
    print(f"\n{json_file.name}")
    print("-" * 40)
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        print(f"Text: {data['text'][:200]}...")
        print(f"Words: {len(data['words'])}")
        print(f"Language: {data['language']}")
        print(f"Model: {data['model']}")